## **IMPORTS**

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import random
import os
from dotenv import load_dotenv

import src.experiments.config as cfg_module
import src.env.station as st
import src.env.car as car_module
import src.env.society as sct
import src.experiments.simulation as sim_module
from src.experiments.run import define_agents

pygame 2.6.1 (SDL 2.28.4, Python 3.12.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


## **UTILS**

### **CONFIG**

In [3]:
# --- CONFIG FUNCTION

def notebook_config(
    total_time=None,
    nb_car=None,
    nb_society=None,
    nb_station=None,
    base_car_behavior=None,
    station_strategy_noise=None,
    log_iter=None
):
    """
    Crée et configure un objet SimulationConfig
    pour les notebooks/tests.
    """

    config = cfg_module.SimulationConfig()

    # ------------------------------------------------ VISUALISATION
    config.set_VISUALIZE(False)

    # ------------------------------------------------ TEMPS
    if total_time is not None:
        config.set_TOTAL_TIME(total_time)

    if log_iter is not None:
        config.set_log_iter(log_iter)

    # ------------------------------------------------ AGENTS
    if nb_car is not None:

        if type(nb_car) is not int:
            raise TypeError(
                f"nb_car doit être un int, reçu : {type(nb_car).__name__}"
            )

        if nb_car <= 0:
            raise ValueError("nb_car doit être > 0")

        config.NB_CARS = nb_car

    if nb_society is not None:

        if type(nb_society) is not int:
            raise TypeError(
                f"nb_society doit être un int, reçu : {type(nb_society).__name__}"
            )

        if nb_society <= 0:
            raise ValueError("nb_society doit être > 0")

        config.NB_SOCIETIES = nb_society

    if nb_station is not None:

        if type(nb_station) is not int:
            raise TypeError(
                f"nb_station doit être un int, reçu : {type(nb_station).__name__}"
            )

        if nb_station <= 0:
            raise ValueError("nb_station doit être > 0")

        config.NB_STATIONS = nb_station

    # ------------------------------------------------ COMPORTEMENTS VOITURES
    if base_car_behavior is not None:

        required_keys = {
            'pres',
            'abs',
            'early',
            'late',
            'noise'
        }

        missing = required_keys - set(base_car_behavior.keys())

        if missing:
            raise ValueError(
                f"Clés manquantes dans base_car_behavior : {missing}"
            )

        total_prob = (
            base_car_behavior['pres']
            + base_car_behavior['abs']
            + base_car_behavior['early']
            + base_car_behavior['late']
        )

        if total_prob != 100:
            raise ValueError(
                f"La somme des probabilités doit valoir 100, reçu : {total_prob}"
            )

        config.BASE_CANCEL_PROB = base_car_behavior.copy()

    # ------------------------------------------------ BRUIT STRATÉGIE STATION
    if station_strategy_noise is not None:

        if not isinstance(station_strategy_noise, (int, float)):
            raise TypeError(
                "station_strategy_noise doit être numérique"
            )

        if station_strategy_noise < 0:
            raise ValueError(
                "station_strategy_noise doit être >= 0"
            )

        config.STRATEGY_NOISE = float(station_strategy_noise)

    return config

In [4]:
# --- EXAMPLE

total_time = 12 * 24 * 1  # une journée
log_iter = 12

nb_car = 50
nb_society = 4
nb_station = 40

base_car_behavior = {
    'pres': 75,     # présent et honore la réservation
    'abs': 10,      # no-show complet
    'early': 9,     # annulation anticipée (> 2h avant)
    'late': 6,      # annulation tardive (< 2h avant)
    'noise': 0.15
}

strategy_noise = 0.5  # variation proportionnelle ±50%

c = notebook_config(total_time=total_time, nb_car=nb_car, nb_society=nb_society,
                nb_station=nb_station, base_car_behavior=base_car_behavior,
                station_strategy_noise=strategy_noise, log_iter=log_iter)

slot_h = c.SLOT_DURATION / 60
dist_slot = c.CAR_SPEED          # m/slot
autonomy_mean = c.CAR_AUTONOMY_PARAMS_KM['mean']
conso = c.ENERGY_CONSUMPTION['quantity_kW'] / c.ENERGY_CONSUMPTION['distance_unit_m']
delta_soc = dist_slot * conso / (c.ENERGY_CONSUMPTION['quantity_kW'] * autonomy_mean / 100)

print(f"Grille            : {c.C_GRID/1e3:.1f} km × {c.C_GRID/1e3:.1f} km")
print(f"Vitesse           : {c.CAR_SPEED} m/slot  ({c.CAR_SPEED/1000/slot_h:.0f} km/h)")
print(f"ΔSoC / slot       : {delta_soc:.5f}  ({1/delta_soc:.0f} slots pour vider)")
print(f"Autonomie moy.    : {autonomy_mean} km")
print(f"Distance max grille traversée avec soc=0.10 : {0.10*autonomy_mean:.0f} km >> {c.C_GRID/1e3:.1f} km ✓")


Grille            : 3.0 km × 3.0 km
Vitesse           : 4167.0 m/slot  (50 km/h)
ΔSoC / slot       : 0.01042  (96 slots pour vider)
Autonomie moy.    : 400 km
Distance max grille traversée avec soc=0.10 : 40 km >> 3.0 km ✓


## **EXPERIMENTS**

### **EVChargingManagement**

#### CONFIG

In [5]:
load_dotenv()
SIM_ID = 'FirstTest'

ROOT_PATH = os.getenv("ROOT_PATH")
OUTPUT_DIR = 'outputs'
os.makedirs(f'{ROOT_PATH}/{OUTPUT_DIR}', exist_ok=True)

#### SIMULATION PARAMS

In [6]:
total_time = 12 * 24 * 1  # une journée
log_iter = 12

nb_car = 50
nb_society = 4
nb_station = 40

base_car_behavior = {
    'pres': 75,     # présent et honore la réservation
    'abs': 10,      # no-show complet
    'early': 9,     # annulation anticipée (> 2h avant)
    'late': 6,      # annulation tardive (< 2h avant)
    'noise': 0.15
}

strategy_noise = 0.5  # variation proportionnelle ±50%

sim_config = notebook_config(total_time=total_time, nb_car=nb_car, nb_society=nb_society,
                nb_station=nb_station, base_car_behavior=base_car_behavior,
                station_strategy_noise=strategy_noise, log_iter=log_iter)


#### INITIALISATION

In [7]:
# ---- OPEN LOG FILES

summary_file = open(f'{ROOT_PATH}/{OUTPUT_DIR}/summary_agents_{SIM_ID}.txt', 'w', 
                    encoding='utf-8') # init file: init state of agents

summary_file.write('\n--------- AGENTS DEFINITION')

summary_file.write("=== Initialisation des agents ===")
cars, stations, societies = define_agents(sim_config)
summary_file.write(f'\n--- {sim_config.NB_CARS} CAR ---')
for car in cars:
    summary_file.write('\n')
    car.display_parameters(file=summary_file)

summary_file.write(f'\n--- {sim_config.NB_SOCIETIES} SOCIETIES ---')
count_society = 1
for society in societies:
    summary_file.write(f'\n --------------- SOCIETY {count_society}/{sim_config.NB_SOCIETIES} ---------------')
    count_society += 1
    society.display_parameters(file=summary_file)
    summary_file.write('\n -------- ATTACHED STATIONS:')
    for station in society.stations:
        summary_file.write('\n')
        station.display_parameters(file=summary_file)

summary_file.write(f"\n=== Simulation started with {len(cars)} cars and {len(stations)} stations. ===")
        

57

#### RUN

In [ ]:
# ----- LANCEMENT SIMULATION

simulation_file = open(f'{ROOT_PATH}/{OUTPUT_DIR}/simulation_{SIM_ID}.txt', 'w', 
                    encoding='utf-8') # init file: init state of agents

simulation_file.write(f"=== Lancement simulation : {sim_config.NB_CARS} voitures, "
        f"{sim_config.NB_STATIONS} stations, {sim_config.TOTAL_TIME} slots ===\n")

simulation = sim_module.Simulation(
    cars=cars,
    stations=stations,
    societies=societies,
    t_max=sim_config.TOTAL_TIME,
    config=sim_config
)
simulation.run(file=simulation_file)

2026-05-24 15:59:12.018 | INFO     | src.experiments.simulation:step:52 - 
--------------------------------------------- INSTANT 0/288---------------------------------------------
2026-05-24 15:59:12.452 | INFO     | src.experiments.simulation:step:52 - 
--------------------------------------------- INSTANT 12/288---------------------------------------------
2026-05-24 15:59:12.571 | INFO     | src.experiments.simulation:step:52 - 
--------------------------------------------- INSTANT 24/288---------------------------------------------
2026-05-24 15:59:12.675 | INFO     | src.experiments.simulation:step:52 - 
--------------------------------------------- INSTANT 36/288---------------------------------------------
2026-05-24 15:59:12.908 | INFO     | src.experiments.simulation:step:52 - 
--------------------------------------------- INSTANT 48/288---------------------------------------------
2026-05-24 15:59:13.041 | INFO     | src.experiments.simulation:step:52 - 
---------------------


========== MÉTRIQUES ==========

--- Station Demand (kWh) ---
  Station 0: 5.77 kWh
  Station 1: 5.92 kWh
  Station 2: 0.00 kWh
  Station 3: 0.00 kWh
  Station 4: 1.47 kWh
  Station 5: 0.00 kWh
  Station 6: 0.00 kWh
  Station 7: 0.00 kWh
  Station 8: 0.00 kWh
  Station 9: 9.47 kWh
  Station 10: 0.00 kWh
  Station 11: 0.00 kWh
  Station 12: 0.00 kWh
  Station 13: 2.50 kWh
  Station 14: 0.00 kWh
  Station 15: 10.18 kWh
  Station 16: 3.22 kWh
  Station 17: 0.47 kWh
  Station 18: 1.43 kWh
  Station 19: 0.00 kWh
  Station 20: 2.13 kWh
  Station 21: 0.00 kWh
  Station 22: 1.98 kWh
  Station 23: 0.00 kWh
  Station 24: 8.88 kWh
  Station 25: 3.48 kWh
  Station 26: 14.60 kWh
  Station 27: 5.59 kWh
  Station 28: 4.71 kWh
  Station 29: 1.20 kWh
  Station 30: 4.97 kWh
  Station 31: 1.05 kWh
  Station 32: 14.32 kWh
  Station 33: 0.00 kWh
  Station 34: 1.88 kWh
  Station 35: 0.00 kWh
  Station 36: 0.00 kWh
  Station 37: 3.85 kWh
  Station 38: 0.00 kWh
  Station 39: 0.64 kWh

--- User Request Satisf

#### DISPLAY